In [102]:
import math
def rpm_to_deg_per_sec(rpm):
    """Converts Rotations Per Minute (RPM) to Degrees Per Second."""
    return rpm * 6.0
def calculate_aoa(x_radar, y_radar, x_aircraft, y_aircraft, aircraft_heading_deg):
    """
    Calculates the Global and Relative Angle of Arrival (AoA) of a radar beam 
    received by an aircraft.
    
    Parameters:
    - x_radar, y_radar: Coordinates of the transmitting radar (e.g., 0, 0)
    - x_aircraft, y_aircraft: Coordinates of the receiving aircraft (e.g., -100, -48)
    - aircraft_heading_deg: Heading angle of the aircraft nose (0 = East, 90 = North)
    """
    # 1. Compute displacement vector pointing FROM aircraft TO radar
    dx = x_radar - x_aircraft
    dy = y_radar - y_aircraft
    
    # 2. Calculate absolute Cartesian Angle of Arrival (East = 0 degrees)
    global_aoa_rad = math.atan2(dy, dx)
    global_aoa_deg = math.degrees(global_aoa_rad) % 360
    
    # 3. Calculate Relative AoA with respect to the aircraft's nose orientation
    relative_aoa_deg = (global_aoa_deg - aircraft_heading_deg) % 360
    
    # Adjust relative angle to standard offset notation (-180 to +180)
    if relative_aoa_deg > 180:
        relative_aoa_offset = relative_aoa_deg - 360
    else:
        relative_aoa_offset = relative_aoa_deg
        
    return relative_aoa_offset
def find_radar_intersections(max_iterations=100,num_hits = 9,AircSpeed = 1, radarRpm = 0.08333):
    # Initial conditions
    x_A = -100
    y_start = -100
    speed_A = AircSpeed
    
    # Radar config (10 RPM = 60 degrees per second)
    # 0 degrees is West (-x axis). Counter-clockwise rotation.
    deg_per_second = rpm_to_deg_per_sec(radarRpm)
    
    intersections = []
    
    print(f"{'Iteration':<10}{'Aircraft X':<12}{'Aircraft Y':<12}{'Aircraft Angle':<16}{'Radar Angle':<12}")
    print("-" * 65)
    
    for t in range(max_iterations):
        # Calculate current aircraft position
        y_A = y_start + (speed_A * t)
        
        # Calculate the true geometric angle of the aircraft from the origin
        # math.atan2(y, x) returns radians. We map it to our custom system:
        # West (-100, 0) is 0 degrees, going counter-clockwise.
        # So we offset by 180 degrees to align math.atan2 with West = 0.
        aircraft_rad = math.atan2(y_A, x_A)
        aircraft_deg = math.degrees(aircraft_rad)
        
        # Normalize aircraft angle to [0, 360) matching counter-clockwise from West
        aircraft_angle_normalized = (aircraft_deg - 180) % 360
        
        # Calculate radar beam angle at this second
        radar_angle = (deg_per_second * t) % 360
        
        # Check if they intersect (allowing a small tolerance for floating-point/discrete matching)
        # In a discrete step simulation, we check if the radar "sweeps past" or hits the aircraft angle
        # For an exact cross, we can check if the difference is close to 0
        angle_diff = abs(aircraft_angle_normalized - radar_angle)
        if angle_diff > 180:
            angle_diff = 360 - angle_diff
            
        # If the radar beam matches the aircraft's angular bearing closely
        if angle_diff < 0.5:  # Tolerance window for discrete sampling
            # Debounce: Ensure we don't log multiple hits for the same pass
            if not intersections or (t - intersections[-1][0]) > 5:
                # Calculate Euclidean distance R from the origin (radar)
                R = math.sqrt(x_A**2 + y_A**2)
                
                # Calculate received power: P = 100 / (4 * pi * R)^2
                power_received = 10**5 / (4 * math.pi * R)**2
                
                intersections.append((t, x_A, y_A, aircraft_angle_normalized, power_received))
                # print(f"{t:<10}{x_A:<12}{y_A:<12}{aircraft_angle_normalized:<16.2f}{radar_angle:<12.2f} *HIT*")

        if len(intersections) == num_hits:
            break
            
    return intersections

# Run the simulation
AircSpeed = 1
radarRpm = 1/12
TOAs = []
Aoa = []
powers = []
Ax = []
Ay = []
hits = find_radar_intersections(max_iterations=6000,num_hits = 10, AircSpeed = AircSpeed, radarRpm = radarRpm)
print("\n--- Final 4 Intersection Points ---")
for i, hit in enumerate(hits, 1):
    print(f"Point {i}: At t={hit[0]}s -> Location: ({hit[1]}, {hit[2]}) at Angle: {hit[3]:.2f} at Power: {hit[4]:.2f}°")
    TOAs.append(hit[0])
    Ax.append(hit[1])
    Ay.append(hit[2])
    Aoa.append(calculate_aoa(0,0,hit[1],hit[2],90))
    powers.append(hit[4])

Iteration Aircraft X  Aircraft Y  Aircraft Angle  Radar Angle 
-----------------------------------------------------------------

--- Final 4 Intersection Points ---
Point 1: At t=52s -> Location: (-100, -48) at Angle: 25.64 at Power: 0.05°
Point 2: At t=564s -> Location: (-100, 464) at Angle: 282.16 at Power: 0.00°
Point 3: At t=1269s -> Location: (-100, 1169) at Angle: 274.89 at Power: 0.00°
Point 4: At t=1986s -> Location: (-100, 1886) at Angle: 273.04 at Power: 0.00°
Point 5: At t=2704s -> Location: (-100, 2604) at Angle: 272.20 at Power: 0.00°
Point 6: At t=3423s -> Location: (-100, 3323) at Angle: 271.72 at Power: 0.00°
Point 7: At t=4142s -> Location: (-100, 4042) at Angle: 271.42 at Power: 0.00°
Point 8: At t=4862s -> Location: (-100, 4762) at Angle: 271.20 at Power: 0.00°
Point 9: At t=5582s -> Location: (-100, 5482) at Angle: 271.05 at Power: 0.00°


In [108]:
import cmath
import numpy as np


def solve_dr_dt(P1, P2, P3, x1, y1, x2, y2, x3, y3):
    """Solves for possible (Dt, Dr) solutions using 3 points of power and position data."""
    # Step 1: Calculate the distance constants d2 and d3
    d2 = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
    d3 = np.sqrt((x3 - x1) ** 2 + (y3 - y1) ** 2)

    # Step 5: Calculate Quadratic Coefficients (A, B, C)
    A = P1 * (P2 - P3)
    B = 2 * P2 * (P1 - P3) * d2 - 2 * P3 * (P1 - P2) * d3 - P1 * (P2 - P3)
    C = P2 * (P1 - P3) * (d2**2) - P3 * (P1 - P2) * (d3**2)

    # Step 6: Solve for Dt using the Quadratic Formula
    discriminant = B**2 - 4 * A * C
    sqrt_disc = cmath.sqrt(discriminant)

    Dt_sol1 = (-B + sqrt_disc) / (2 * A)
    Dt_sol2 = (-B - sqrt_disc) / (2 * A)

    solutions = []

    # Step 7: Substitute each Dt back to solve for Dr
    for Dt in [Dt_sol1, Dt_sol2]:
        # Calculate Dr^2 from Equation 1
        Dr2 = (P2 * (Dt + d2) ** 2 - P1 * Dt) / (P1 - P2)
        Dr_pos = cmath.sqrt(Dr2)
        Dr_neg = -cmath.sqrt(Dr2)

        solutions.append((Dt, Dr_pos))
        solutions.append((Dt, Dr_neg))

    return solutions
def filter_valid_solution(solutions, tol=1e-5, Aoa=0):
    """Calculates the angular error for all solutions and returns the one

    with the absolute lowest error relative to the target AoA.
    """
    if not solutions:
        raise ValueError("The solutions list is empty.")

    evaluated_solutions = []

    # Step 1: Calculate angle_diff for ALL solutions
    for Dt, Dr in solutions:
        r_Dt = Dt.real
        r_Dr = Dr.real
        print(r_Dt,r_Dr)

        # Calculate geometric angle
        deg = math.degrees(math.atan2(r_Dt, r_Dr))
        if deg > 180:
            deg = deg - 360

        # Calculate modular angular difference (shortest path on a circle)
        angle_diff = abs(deg - Aoa) % 360
        if angle_diff > 180:
            angle_diff = 360 - angle_diff

        # Store the calculated difference alongside the components
        evaluated_solutions.append((angle_diff, r_Dr, r_Dt))

    # Step 2: Find and return the one with the least angle_diff
    # min() will sort by the first element of the tuple (angle_diff)
    least_error_sol = min(evaluated_solutions, key=lambda x: x[0])

    # Extract components of the best match
    best_angle_diff, final_Dr, final_Dt = least_error_sol

    return final_Dr, final_Dt


def calculate_radar_rpm_from_power(
    P1,
    P2,
    P3,
    P4,
    x1,
    y1,
    x2,
    y2,
    x3,
    y3,
    x4,
    y4,
    t1,
    t2,
    Aoa_1,
    Aoa_2,
    Aoa_3,
    Aoa_4,
    ac_speed,
):
    """Calculates radar RPM by solving for positions at two main-lobe intervals."""
    # 1. Calculate Dr1, Dt1 using points 1, 2, 3
    sols_1 = solve_dr_dt(P1, P2, P3, x1, y1, x2, y2, x3, y3)
    Dr1, Dt1 = filter_valid_solution(sols_1,Aoa_1)

    print(f"Selected Point 1 Parameters -> Dr1: {Dr1:.2f} m, Dt1: {Dt1:.2f} m")

    # 2. Calculate Dr2, Dt2 using points 2, 3, 4
    print((P2, P3, P4, x2, y2, x3, y3, x4, y4))
    sols_2 = solve_dr_dt(P2, P3, P4, x2, y2, x3, y3, x4, y4)
    Dr2, Dt2 = filter_valid_solution(sols_2,Aoa_2)
    print(f"Selected Point 2 Parameters -> Dr2: {Dr2:.2f} m, Dt2: {Dt2:.2f} m\n")

    # 3. Calculate time delta between peak 1 and peak 2
    dt = t2 - t1
    if dt <= 0:
        raise ValueError("t2 must be greater than t1")

    # 4. Calculate straight line range to radar at both peak instances
    R1 = np.sqrt(Dr1**2 + Dt1**2)
    R2 = np.sqrt(Dr2**2 + Dt2**2)

    # 5. Distance traveled by aircraft between the two main peaks
    d_ac = ac_speed * dt

    # 6. Law of cosines to isolate angular change relative to the radar position
    cos_theta = (R1**2 + R2**2 - d_ac**2) / (2 * R1 * R2)
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    delta_theta_ac = np.arccos(cos_theta)

    # 7. Total angular rotation of radar main beam
    theta_radar = 2 * np.pi + delta_theta_ac

    # 8. Compute Angular velocity and convert to RPM
    omega = theta_radar / dt
    rpm = (omega * 60) / (2 * np.pi)

    return rpm

In [109]:
for i in range(len(TOAs)):
    if(i<4):
        print(TOAs[i],powers[i],Ax[i],Ay[i],Aoa[i])

52 0.05146760384952951 -100 -48 -64.3589941756947
564 0.00281077958669755 -100 464 -167.83779648031816
1269 0.00046002857684084556 -100 1169 -175.11063872303464
1986 0.00017753241039928592 -100 1886 -176.96488988011572


In [110]:
radar_rpm = calculate_radar_rpm_from_power(
    powers[0],
    powers[1],
    powers[2],
    powers[3],
    Ax[0],
    Ay[0],
    Ax[1],
    Ay[1],
    Ax[2],
    Ay[2],
    Ax[3],
    Ay[3],
    TOAs[0],
    TOAs[1],
    Aoa[0],
    Aoa[1],
    Aoa[2],
    Aoa[3],
    AircSpeed,
)
print(radar_rpm)

-51.54970362052352 110.91464519010809
-51.54970362052352 -110.91464519010809
-710.5905290800275 55.0443710925013
-710.5905290800275 -55.0443710925013
Selected Point 1 Parameters -> Dr1: 110.91 m, Dt1: -51.55 m
(0.00281077958669755, 0.00046002857684084556, 0.00017753241039928592, -100, 464, -100, 1169, -100, 1886)
314.1279018962038 450.4184091666073
314.1279018962038 -450.4184091666073
-969.4355942038958 121.83309996760138
-969.4355942038958 -121.83309996760138
Selected Point 2 Parameters -> Dr2: 450.42 m, Dt2: 314.13 m

0.13871258238059908
